# Connect and Login

In [1]:
import win32com.client

connection_name = 'JGO_CO1SQLWPV22'
user_name = ''
password=''

ResQApp = win32com.client.Dispatch("ResQ3Automation.ResQApplication")
ResQApp.ConnectByName('JGO_CO1SQLWPV22', user_name, password) # use window authentication

In [5]:
ResQApp.Disconnect() # Release license after work complete

# Select project and reserving class path

In [2]:
ProjectName = 'NJ_Annual_Prod_202605_Fake'
# Path = r'PRNJ - PA\PA\All States\Direct Group\CMPxCAT'
Path = r"PRNJ - PA\PA\NY\Direct Group\MP+PIP"
Path = r"HPPREF\HO+DF\NJ\Legacy\HOL"

project = ResQApp.Projects().Item(ProjectName)
reserving_class = project.ReservingClasses().Item(Path)

# Datasets

## Triangles

In [86]:
TriangleName = 'ALAE--Paid'
triangle = reserving_class.Triangles().Item(TriangleName)

# DataFormat: 0=Triangle
triangle.Name
triangle.DatasetType.DataFormat

0

In [87]:
from datetime import datetime

triangle.OriginLabel(1) # use the first row label, count the development columns
triangle.DevelopmentCount(OriginDate=datetime(int(triangle.OriginLabel(1)), 12, 31))

10

In [42]:
top_n = 3
count = 0

for i in reserving_class.Triangles():
    print([i.Name, i.DatasetType.Name, i.DatasetType.DataFormat, i.User, i.Created, i.Modified])
    count += 1
    if count >= top_n:
        break

['Claim Counts--Reported', 'Claim Counts--Reported', 0, 'Wei, Xiao ADMIN', pywintypes.datetime(2023, 2, 3, 9, 27, 6, 761000, tzinfo=TimeZoneInfo('GMT Standard Time', True)), pywintypes.datetime(2026, 3, 2, 16, 28, 51, 235000, tzinfo=TimeZoneInfo('GMT Standard Time', True))]
['Recoveries--Received', 'Recoveries--Received', 0, 'Wei, Xiao ADMIN', pywintypes.datetime(2023, 2, 3, 10, 28, 6, 86000, tzinfo=TimeZoneInfo('GMT Standard Time', True)), pywintypes.datetime(2026, 3, 2, 17, 28, 42, 47000, tzinfo=TimeZoneInfo('GMT Standard Time', True))]
['Gross Loss--OS', 'Gross Loss--OS', 0, 'Wei, Xiao ADMIN', pywintypes.datetime(2023, 2, 3, 10, 28, 3, 348000, tzinfo=TimeZoneInfo('GMT Standard Time', True)), pywintypes.datetime(2026, 3, 2, 17, 30, 50, 457000, tzinfo=TimeZoneInfo('GMT Standard Time', True))]


## Vectors

In [6]:
[i.Name for i in reserving_class.Vectors() if '91' in i.Name]

['C 91 -  Current Qtr Indicated',
 'F 91 - Current Qtr Indicated',
 'G 91 - Current Qtr Indicated ',
 'F 91 - Current Qtr Indicated - Feb 2026',
 'C 91 -  Current Qtr Indicated - Feb 2026',
 'G 91 - Current Qtr Indicated  - Feb 2026',
 'G 91 - Current Qtr Indicated  - Feb 2026']

In [14]:
VectorName = 'C 41 - BF Reported ex CWOP'
VectorName = 'C 61 Reported - CWOP'
vector = reserving_class.Vectors().Item(VectorName)

# DataFormat: 1=Vector
vector.DatasetType.DataFormat

1

In [15]:
vector.Formula

'"C 32 - Reported DFM w/ Selected LDFs   " - "C 22 - CWOP DFM w/ Selected LDFs  "'

### Read and write vector values


In [ ]:
# ResQ vectors are one-dimensional datasets. Values are addressed by 1-based origin index.
origin_index = 1

vector.Count
vector.OriginLabel(origin_index)
vector.ValuesByIndex(origin_index)


In [ ]:
# Write one vector value back to ResQ. Use with care on a scratch/test project.
# new_value = vector.ValuesByIndex(origin_index)
# vector.SetValuesByIndex(origin_index, new_value)
# vector.Save()


In [ ]:
# ResQ method type codes used by vector.MethodType / OutputVector.MethodType
METHOD_TYPES = {
    0: "None",
    1: "DFM",
    2: "BF",
    3: "CC",
    4: "Result Selection",
}

vector.MethodType
METHOD_TYPES.get(vector.MethodType, vector.MethodType)


In [ ]:
vector.Method.Name 

In [ ]:
vector.MethodType
# MethodType: 0=None, 1=DFM, 2=BF, 3=CC, 4=Result Selection


# DFM method properties

## Read properties

In [10]:
# In ResQ, some instance name and dataset type name have unncessary white space(s) inside/after the name, 
# during the transition, we need to trim those white spaces and use a clean version and standarized version of the names
all_DFMs = list(i.Name for i in reserving_class.DFMMethods())  # always use reserving_class.DFMMethods, not project.DFMMethods
all_DFMs

['F 25 - Incurred DFM Bootstrap',
 'C 22 - CWOP DFM w/ Selected LDFs  ',
 'C 32 - Reported DFM w/ Selected LDFs   ',
 'F 23 - Incurred DFM w/ Selected LDFs ',
 'F 22 - Incurred DFM w/ Prior LDFs',
 'G 11 - ALAE--Paid DFM w/ Prior LDFs ',
 'G 12 - ALAE--Paid DFM w/ Selected LDFs  ',
 'C 12 - CWP DFM w/ Selected LDFs ',
 'C 52 - CWOP/Reported DFM w/ Selected LDFs  ',
 'C 42 - Reported ex CWOP DFM w/ Selected LDFs  ',
 'F 12 - Paid DFM w/ Prior LDFs',
 'F 13 - Paid DFM w/ Selected LDFs ',
 'H 12 - Net Incurred per Reported ex CWOP DFM w/ Prior LDFs',
 'H 02 - Net Incurred per Reported ex CWOP DFM w/ Selected LDFs   ',
 'G 21 - ALAE/Paid Loss DFM w/ Prior LDFs  ',
 'G 22 B - ALAE/Net Paid Loss DFM w/ Selected LDFs',
 'G 43 - DFM for BF Paid',
 'F 43 - DFM for BF Incurred',
 'F 46 - DFM for BF Incurred',
 'F 23 C - Adjusted Incurred DFM w/ Selected LDFs',
 'F 23 G - Adjusted Incurred DFM',
 'F 28 - BS Incurred DFM',
 'F 43 C - Adjusted Incurred DFM for BF']

In [10]:
# Development Factor Method (DFM) ... The api method name is DFMMethods ...
DFM_MethodName = r'C 12 - CWP DFM w/ Selected LDFs'

aDFM = reserving_class.DFMMethods().Item(DFM_MethodName)

org_rng = range(1, aDFM.OriginCount+1)
dev_rng = range(1, aDFM.DevelopmentCount(1)+1)

aDFM.OriginLength
aDFM.DevelopmentLength

aDFM.OriginCount # number of rows
aDFM.DevelopmentCount(1) # number of dev cols (look at the first row)

# aDFM.ExcludedRatios(i, j) 
# return values 0, 1, 2 for triangle cell i, j
# 0=included; 
# 1=excluded; 
# 2=empty cell (no value)

# ratio selection status (pattern)
excluded_ratio_pattern = [[aDFM.ExcludedRatios(i, j) for j in dev_rng] for i in org_rng]

# Other Attributes
try:
    aDFM.SummaryRatioBasis.Name # ratio_basis_dataset
except:
    print('Ratio Basis not Selected')
aDFM.RatioDecimalPlaces
aDFM.SummaryRatioDecimalPlaces
aDFM.InputTriangle.Name
aDFM.OutputVector.Name
aDFM.OutputVector.Modified # returns pywintypes.datetime(yyyy, m, d, h, m, s, ..., tzinfo=TimeZoneInfo('GMT Standard Time', True))

average_formulas = [aDFM.AverageFormula(i) for i in range(1, 15)] # get all average formula names

# capped the max row at 20 since all values after 13 will generally be 'XX: User Entry'
# remove "<index>: " before the actual formula name
# only allow one User Entry stored in json file, 

for idx in range(20):
    idx_name = f'{idx}: User Entry'
    if idx_name in average_formulas:
        first_entry_idx = average_formulas.index(idx_name)
        break
        
# pick the first User Entry and the stored values for each development period (column)
user_entry_values = [aDFM.AverageRatioValues(j, first_entry_idx+1) for j in org_rng]

# find the selected average formula index
[aDFM.SelectedRatios(DevIndex=j) for j in dev_rng]

Ratio Basis not Selected


[5, 5, 5, 5, 5, 5, 5, 5, 5, 5]

In [19]:
aDFM.OutputVector.Modified

pywintypes.datetime(2026, 6, 5, 15, 40, 43, 468000, tzinfo=TimeZoneInfo('GMT Standard Time', True))

In [12]:
?aDFM.SetUserRatios

Signature:
aDFM.SetUserRatios(
    DevIndex=<PyOleMissing object at 0x0000014D81502AE0>,
    AvgIndex=<PyOleMissing object at 0x0000014D81502AE0>,
    arg2=<PyOleMissing object at 0x0000014D81502AE0>,
)
Docstring: <no docstring>
File:      e:\arcrho server\library\<comobject item>
Type:      method

In [13]:
?aDFM.SetSelectedRatios

Signature:
aDFM.SetSelectedRatios(
    DevIndex=<PyOleMissing object at 0x0000014D81502AE0>,
    arg1=<PyOleMissing object at 0x0000014D81502AE0>,
)
Docstring: <no docstring>
File:      e:\arcrho server\library\<comobject item>
Type:      method

In [17]:
final_value = 8.8888  # growth adjustments applied
# Normally, user entry is the 10th row
aDFM.SetUserRatios(1, 10, final_value)
aDFM.SetSelectedRatios(1, 10)
aDFM.save()

In [14]:
aDFM.Ratios(OriginIndex=1, DevIndex=1)

4.8614176514337855

In [15]:
[aDFM.OriginLabel(i) for i in dev_rng]

['2017',
 '2018',
 '2019',
 '2020',
 '2021',
 '2022',
 '2023',
 '2024',
 '2025',
 '2026']

In [13]:
[aDFM.DevelopmentLabel(i) for i in dev_rng]

['(1) 2-14',
 '(2) 14-26',
 '(3) 26-38',
 '(4) 38-50',
 '(5) 50-62',
 '(6) 62-74',
 '(7) 74-86',
 '(8) 86-98',
 '(9) 98-110',
 '(10) 110-122']

In [21]:
[aDFM.Ultimates(i) for i in org_rng]

[0.6742911871006106,
 0.7959666538857079,
 0.7698121801868977,
 0.822055433818667,
 0.6636776499689283,
 0.6873301212567179,
 0.6188120799338108,
 0.6122442229741147,
 0.72323353475119,
 0.27747444199632876]

In [17]:
aDFM.CellNotes

'"Ratios.Ratios & Average Selection", Cell[2-14, 2025], "note test xxx ratio", User: Wei, Xiao, Date: 5/28/2026\r\n"Ratios.Ratios & Average Selection", Cell[2-14, Simple - 8], "note 2", User: Wei, Xiao, Date: 5/28/2026\r\n'

In [14]:
print(aDFM.CellNotes)
# Tab, X Label, Y Label, User, Date, Notes  -- The labels in ResQ UI
# The value for X Label shows 2-14 in aDFM.CellNotes means '(1) 2-14' from aDFM.DevelopmentLabel(1)

"Ratios.Ratios & Average Selection", Cell[2-14, 2025], "note test xxx ratio", User: Wei, Xiao, Date: 5/28/2026
"Ratios.Ratios & Average Selection", Cell[2-14, Simple - 8], "note 2", User: Wei, Xiao, Date: 5/28/2026



## Write DFM data into ResQ

In [ ]:
aDFM.SetExcludedRatios(OriginIndex=1, DevIndex=1, arg2=1)  # set cell (1, 1) as excluded
aDFM.SetExcludedRatios(OriginIndex=1, DevIndex=1, arg2=0)  # set cell (1, 1) as included (not excluded)

aDFM.SetSelectedRatios(DevIndex=1, arg1=2)  # set the selected average formula index to be 2 (2nd formula) for development column 1  

aDFM.Notes = 'New Notes'  # need to use /r/n to change line instead of /n

aDFM.Save()  # Save all changes to real Database

# Berquist Sherman

In [ ]:
# Berquist Sherman (bs)
# Settlement Rate (sr)
bs_sr_name = 'Gross Loss--Paid  - B&S Settlement Rate Adjustment' # bs triangle (with method attached) instance name

In [ ]:
bs_method = reserving_class.GetBerquistShermanSR(bs_sr_name)

In [ ]:
# "Paid Loss" (Input Triangle 1)
bs_method.PaidClaims.Name

# "Closed Claim Counts" (Input Triangle 2)
bs_method.ClosedClaimNos.Name

# "Ultimate Claim Counts" (Input Vector)
bs_method.UltimateClaimNos.Name

# Output Triangle Dataset Type
bs_method.OutputTriangle.DatasetType.Name

bs_method.OriginLength # int
bs_method.DevelopmentLength # int

In [ ]:
output_tri = bs_method.OutputTriangle

In [ ]:
output_tri.ValuesByIndex(1,1)

# Result Selection

In [ ]:
ResultSelectionName = 'C 91 -  Current Qtr Indicated'
result_selection = reserving_class.GetResultSelection(ResultSelectionName)

# Result Selection output vectors use MethodType 4.
result_selection.OutputVector.Name
result_selection.OutputVector.DatasetType.Name
result_selection.OutputVector.MethodType


In [ ]:
# Basic shape and labels
result_selection.OriginLength
result_selection.OriginCount
result_selection.DatasetCount
result_selection.OriginLabel(1)


In [ ]:
# Source datasets used by the Result Selection method. Dataset(i) is 1-based.
source_index = 1
source_dataset = result_selection.Dataset(source_index)

source_dataset.Name
source_dataset.DatasetType.Name
source_dataset.DatasetType.DataFormat  # 0=Triangle, 1=Vector


In [ ]:
# Read selected source values, weights, and selected ultimate by origin row.
origin_index = 1
origin_length = result_selection.OriginLength

source_value = result_selection.DatasetValues(source_index, origin_index, origin_length)
source_weight = result_selection.Weights(source_index, origin_index)
selected_ultimate = result_selection.Ultimates(origin_index, origin_length)

source_value, source_weight, selected_ultimate


In [ ]:
# Build a compact table of source values and weights for all origins.
rows = []
for row_index in range(1, result_selection.OriginCount + 1):
    row = {
        "origin": result_selection.OriginLabel(row_index),
        "selected_ultimate": result_selection.Ultimates(row_index, result_selection.OriginLength),
    }
    for dataset_index in range(1, result_selection.DatasetCount + 1):
        dataset_name = result_selection.Dataset(dataset_index).Name
        row[f"{dataset_name} value"] = result_selection.DatasetValues(dataset_index, row_index, result_selection.OriginLength)
        row[f"{dataset_name} weight"] = result_selection.Weights(dataset_index, row_index)
    rows.append(row)

rows[:3]


In [ ]:
# Write one Result Selection weight back to ResQ. Use with care on a scratch/test project.
# result_selection.SetWeights(source_index, origin_index, source_weight)
# result_selection.Save()
